In [1]:
import pandas as pd
import re
import json
from pathlib import Path

# ========= Paths =========
input_files = [
    Path("knowledge/authoritative_kb_primekg.json"),
    Path("knowledge/authoritative_mayo_kb.json"),
    Path("knowledge/authoritative_webmd_kb.json")
]

output_path = Path("dataset/allknowledge.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

# ========= Clean text =========
def clean_text(text):
    if pd.isna(text):
        return None

    text = str(text)

    # 去掉多余换行、回车、制表符
    text = re.sub(r"[\r\n\t]+", " ", text)

    # 去掉特殊符号（保留字母、数字、基本标点）
    text = re.sub(r"[^\w\s,.!?;:'\"()\-/%]", " ", text)

    # 去掉重复空格
    text = re.sub(r"\s+", " ", text).strip()

    # 清洗后为空则记为缺失值
    return text if text != "" else None

# ========= Parse PrimeKG =========
def load_primekg_json(file_path):
    """
    authoritative_kb_primekg.json 的结构大致是：
    {
        "Metformin": ["indication type 2 diabetes mellitus", ...],
        ...
    }

    为了不改后续 LLM 调用逻辑，这里把每一条关系拼成一条 text：
    "Metformin indication type 2 diabetes mellitus"
    """
    rows = []

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, dict):
        raise ValueError(f"{file_path} should be a dict.")

    for subject, relations in data.items():
        if isinstance(relations, list):
            for rel in relations:
                if rel is None:
                    continue
                text = f"{subject} {rel}"
                rows.append({"text": text})
        elif relations is not None:
            text = f"{subject} {relations}"
            rows.append({"text": text})

    return rows

# ========= Parse Mayo / WebMD =========
def load_nested_kb_json(file_path):
    """
    authoritative_mayo_kb.json 和 authoritative_webmd_kb.json 的结构大致是：
    [
      {
        "title": "...",
        "content": {
            "Section A": ["sentence1", "sentence2", ...],
            ...
        }
      },
      ...
    ]

    这里把每一个句子/段落都整理成一条 text。
    可附带 title 和 section，方便给 LLM 更多上下文，但仍然只保留到 text 列里。
    """
    rows = []

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"{file_path} should be a list.")

    for doc in data:
        if not isinstance(doc, dict):
            continue

        title = doc.get("title", "")
        content = doc.get("content", {})

        if not isinstance(content, dict):
            continue

        for section, items in content.items():
            if isinstance(items, list):
                for item in items:
                    if item is None:
                        continue
                    text = f"{title}. {section}. {item}"
                    rows.append({"text": text})
            elif items is not None:
                text = f"{title}. {section}. {items}"
                rows.append({"text": text})

    return rows

# ========= Load all files =========
all_rows = []

for file_path in input_files:
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    if "primekg" in file_path.name.lower():
        rows = load_primekg_json(file_path)
    else:
        rows = load_nested_kb_json(file_path)

    all_rows.extend(rows)

# ========= Create DataFrame =========
df = pd.DataFrame(all_rows)

# ========= Keep only the "text" column =========
df = df[["text"]].copy()

# ========= Clean =========
df["text"] = df["text"].apply(clean_text)

# ========= Remove null values =========
df = df.dropna(subset=["text"]).reset_index(drop=True)

# ========= Save as CSV =========
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("清洗完成，文件已保存到：", output_path)
print("保留行数：", len(df))

清洗完成，文件已保存到： dataset/allknowledge.csv
保留行数： 5281
